In [1]:
from typing import TypedDict, Optional, Dict, Any
from langgraph.graph import StateGraph, END
from src.risk import compute_risk_snapshot
from src.macro import check_macro_surprise
from src.news import get_news_sentiment
from src.news import client

In [2]:
class PipelineState(TypedDict):
    tickers: list
    weights: list
    keywords: list
    macro_result: Optional[Dict[str, Any]]
    news_result: Optional[Dict[str, Any]]
    risk_result: Optional[Dict[str, Any]]
    memo: Optional[str]

In [3]:
def macro_check(state: PipelineState) -> PipelineState:
    print("Running macro check...")
    result = check_macro_surprise()
    return {"macro_result": result}

In [4]:
def news_check(state: PipelineState) -> PipelineState:
    print("Running news check...")
    result = get_news_sentiment(tickers=state["tickers"], keywords=state["keywords"])
    return {"news_result": result}

In [5]:
def risk_check(state: PipelineState) -> PipelineState:
    print("Running risk check...")
    result = compute_risk_snapshot(tickers=state["tickers"], weights=state["weights"])
    return {"risk_result": result}

In [6]:
def route_after_checks(state: PipelineState) -> str:
    macro_flag = state["macro_result"] is not None
    news_flag = state["news_result"] is not None and abs(state["news_result"]["average_sentiment"]) > 0.1

    if macro_flag or news_flag:
        print("Something's flagged — running full risk check")
        return "risk_check"
    else:
        print("Nothing flagged — skipping risk check")
        return END

In [7]:
def synthesize_memo(state: PipelineState) -> PipelineState:
    print("Synthesizing memo...")

    prompt = f"""You are a portfolio risk analyst. Write a short risk memo (3-4 sentences) based on the following data.

Macro surprise: {state["macro_result"]}
News sentiment: {state["news_result"]}
Risk metrics: {state["risk_result"]}

Explain what changed, the portfolio's current exposure, and a suggested action.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )

    memo_text = response.choices[0].message.content
    return {"memo": memo_text}

In [8]:
graph = StateGraph(PipelineState)

graph.add_node("macro_check", macro_check)
graph.add_node("news_check", news_check)
graph.add_node("risk_check", risk_check)
graph.add_node("synthesize_memo", synthesize_memo)

graph.set_entry_point("macro_check")
graph.add_edge("macro_check", "news_check")
graph.add_conditional_edges("news_check", route_after_checks)
graph.add_edge("risk_check", "synthesize_memo")
graph.add_edge("synthesize_memo", END)

app = graph.compile()

In [9]:
result = app.invoke({
    "tickers": ["CDSL.NS", "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS"],
    "weights": [0.25, 0.25, 0.25, 0.25],
    "keywords": ["CDSL", "Reliance", "RIL", "TCS", "HDFC Bank", "HDFC"],
    "macro_result": None,
    "news_result": None,
    "risk_result": None,
    "memo": None
})

print("\n--- FINAL RESULT ---")
print(result.get("memo", "No memo generated"))

Running macro check...
Running news check...
Something's flagged — running full risk check
Running risk check...


[*********************100%***********************]  4 of 4 completed


Synthesizing memo...

--- FINAL RESULT ---
**Risk Memo – Current Outlook**  
With no macro surprise and a mildly bearish news tone (average sentiment = ‑0.15), the portfolio’s downside risk has sharpened: 95 % VaR is now ≈ 1.96 % of NAV, the Sharpe ratio has slipped into negative territory (‑0.99), and the max drawdown sits at‑24.8 %. Correlation analysis shows a moderate concentration to the financial and energy sectors (e.g., 0.54 × HDFCBANK and 0.43 × RELIANCE), amplifying exposure to the recent negative headlines around Reliance and Airtel. Given the deteriorating risk‑adjusted return and sector‑specific headwinds, we recommend trimming the Reliance‑heavy exposure and rebalancing
